<h1>EAR AND MAR Computations</h1>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

**Euclidean Distance**

In [ ]:
def euclidean(df, i, j):
    return np.sqrt((df[f'x{i}'] - df[f'x{j}'])**2 + (df[f'y{i}'] - df[f'y{j}'])**2)

**EAR and MAR Implementation**

In [ ]:
def compute_EAR(df):
    r_ear = (euclidean(df,160,144) + euclidean(df,158,154)) / (2 * euclidean(df,33,133))
    l_ear = (euclidean(df,385,381) + euclidean(df,387,373)) / (2 * euclidean(df,362,263))
    return (r_ear + l_ear) / 2

def compute_MAR(df):
    vertical = (euclidean(df, 13,14) + euclidean(df, 82,87) + euclidean(df, 312,317))
    horizontal = euclidean(df, 78,308)
    return vertical / (2 * horizontal)

**Visualization for documentation**

In [ ]:
def visualize_MAR(df, setup_num):
    legend_elements = [Patch(facecolor='green', label='Alert'),
                    Patch(facecolor='orange', label='Drowsy'),
                    Patch(facecolor='red', label='Microsleep')]

    color_map = {'alert': 'green', 'drowsy': 'orange', 'microsleep': 'red'}
    colors = df['class'].map(color_map)

    plt.figure(figsize=(18, 5))

    plt.plot(df.index, df['MAR'], color='black', linewidth=0.5, alpha=0.7, zorder=2)

    for i, (idx, row) in enumerate(df.iterrows()):
        plt.axvspan(idx, idx + 1, color=color_map[row['class']], alpha=0.15, linewidth=0)

    plt.xlabel('Frame Index')
    plt.ylabel('MAR')
    plt.title(f'MAR vs Frame Index (setup_{setup_num})')
    plt.legend(handles=legend_elements)
    plt.tight_layout()
    plt.savefig(f'..\\Logs\\setup_{setup_num}_MAR')
    plt.show()

In [ ]:
def visualize_EAR(df, setup_num):
    legend_elements = [Patch(facecolor='green', label='Alert'),
                    Patch(facecolor='orange', label='Drowsy'),
                    Patch(facecolor='red', label='Microsleep')]

    color_map = {'alert': 'green', 'drowsy': 'orange', 'microsleep': 'red'}
    colors = df['class'].map(color_map)

    plt.figure(figsize=(18, 5))

    plt.plot(df.index, df['EAR'], color='black', linewidth=0.5, alpha=0.7, zorder=2)

    for i, (idx, row) in enumerate(df.iterrows()):
        plt.axvspan(idx, idx + 1, color=color_map[row['class']], alpha=0.15, linewidth=0)

    plt.xlabel('Frame Index')
    plt.ylabel('EAR')
    plt.title(f'EAR vs Frame Index (setup_{setup_num})')
    plt.legend(handles=legend_elements)
    plt.tight_layout()
    plt.savefig(f'..\\Logs\\setup_{setup_num}_EAR')
    plt.show()

**Add to CSV**

In [ ]:
# setup_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
setup_list = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

for setup_num in setup_list:
    print("Processing setup", setup_num)
    CSV_PATH   = f'..\\CSV Dataset\\landmarks_setup_{setup_num}.csv'

    df = pd.read_csv(CSV_PATH)

    df['EAR'] = compute_EAR(df)
    df['MAR'] = compute_MAR(df)

    df = df.drop(['time'], axis=1)
    df.to_csv(CSV_PATH, index=False)

    visualize_MAR(df, setup_num)
    visualize_EAR(df, setup_num)

    print("Done processing setup", setup_num)

raise SystemExit()

**Debugging**

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

VIDEO_PATH = r"..\\Video Dataset\\setup_1\\face_setup_1.mp4"  # any setup video
# FRAME_TO_CHECK = 1435  # yawn frame
FRAME_TO_CHECK = 100  # alert frame


# indices_to_check = [33, 160, 158, 133, 153, 144,    # right eye
#                     362, 385, 387, 263, 373, 380]    # left eye

indices_to_check = [13, 14, 80, 88, 312, 317, 78, 308]

mp_face_mesh = mp.solutions.face_mesh

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_TO_CHECK)
ret, frame = cap.read()
cap.release()

if not ret:
    print("Could not read frame")
    exit()

h, w = frame.shape[:2]

with mp_face_mesh.FaceMesh(static_image_mode=True) as face_mesh:
    results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark

        for idx in indices_to_check:
            x = int(landmarks[idx].x * w)
            y = int(landmarks[idx].y * h)
            cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)
            cv2.putText(frame, str(idx), (x + 4, y - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 0), 1)

cv2.imwrite("eye_landmarks_check.png", frame)
print("Saved eye_landmarks_check.png")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(CSV_PATH).iloc[1430:1440]

def euclidean(df, i, j):
    return np.sqrt((df[f'x{i}'] - df[f'x{j}'])**2 + (df[f'y{i}'] - df[f'y{j}'])**2)

horiz = euclidean(df, 78, 308)
vert1 = euclidean(df, 13, 14)
vert2 = euclidean(df, 82, 87)
vert3 = euclidean(df, 312, 317)

print("Horizontal (78-308):", horiz.values)
print("Vertical 1 (13-14):", vert1.values)
print("Vertical 2 (82-87):", vert2.values)
print("Vertical 3 (312-317):", vert3.values)
print("MAR:", ((vert1 + vert2 + vert3) / (2 * horiz)).values)
